In [1]:
import os 
import pandas as pd
import pyiceberg
import pyarrow
import warnings
from contextlib import suppress
from pyiceberg.catalog import load_catalog

warnings.filterwarnings("ignore") 


catalog = load_catalog(
    uri=os.environ.get("ICEBERG_REST", "http://127.0.0.1:3005"),
)

# load source table
source = catalog.load_table("public.cbrf_money_columns")

# create sink table
with suppress(pyiceberg.exceptions.NoSuchTableError):
    catalog.drop_table('public.cbrf_money_processing')
sink = catalog.create_table('public.cbrf_money_processing', source.schema())

with sink.update_schema() as tx:
    tx.add_column("year", pyiceberg.schema.LongType())
sink_schema = sink.scan(row_filter="FALSE").to_arrow().schema

# process data in batches
rbr = source.scan().to_arrow_batch_reader()
for batch in rbr:
    batch: pyarrow.RecordBatch
    
    print ('append batch')
    part_df = batch.to_pandas()
    part_df["year"] = pd.to_datetime(part_df['date']).dt.year
    
    sink_part = pyarrow.Table.from_pandas(part_df, schema=sink_schema)
    sink.append(sink_part)
    
print (sink.scan().count())

In [1]:
select year, *
from public.cbrf_money_processing
